# Practice Lab: Conversational RAG (Lesson 8.8)

Module 8 · 8 exercises · Reformulation + memory + token budgeting + multi-turn eval

Exercises 1, 4, 5, 7 run locally (pure Python).


# Lesson 8.8: Conversational RAG — Memory + Retrieval

8 exercises: 2E + 3M + 3C

Exercises 1, 4, 5, 7 run **locally** (pure Python). Ex 2, 3, 6, 8 are architecture/design.


In [ ]:
import numpy as np
import hashlib


---
## Exercise 1 (Easy): Query Reformulation


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib
import numpy as np

def fe(t, dim=64):
    h = hashlib.sha256(t.lower().encode()).digest()
    v = np.array([b/255.0 for b in h[:dim]])
    for w, i in {"refund":0,"policy":1,"price":2,"cost":3,"course":4,"genai":5,"prereq":6,"python":7,"emi":8,"live":10}.items():
        if w in t.lower() and i < dim: v[i] += 0.5
    return v / np.linalg.norm(v)

def cosine(a, b): return np.dot(a, b)/(np.linalg.norm(a)*np.linalg.norm(b))

docs = ["GenAI course costs 14999 rupees with lifetime access",
        "Refund policy: full refund within 7 days of purchase",
        "Prerequisites: basic Python and high school math",
        "EMI available via Razorpay starting 2500 per month",
        "Live classes daily 7 PM IST on YouTube recordings"]
dembs = [fe(d) for d in docs]

def retrieve(q):
    qe = fe(q); best = max(range(len(docs)), key=lambda i: cosine(qe, dembs[i]))
    return docs[best], round(cosine(fe(q), dembs[best]), 3)

def reformulate(fu, history):
    if not history: return fu
    topics = set()
    for m in history[-2:]:
        for w in m["content"].lower().split():
            if w not in {"what","how","does","the","is","are","a","an","i","it","that","this","can","do","much","about"}: topics.add(w)
    for pronoun in ["it","that","this","they","its"]:
        if pronoun in fu.lower():
            return fu.lower().replace(pronoun, f"the {' '.join(list(topics)[:3])}")
    return fu

history = []
turns = [("What courses does Netsetos offer?", True), ("How much does it cost?", False),
         ("Can I pay in installments?", False), ("What if I don't like it?", False),
         ("What are the prerequisites?", False), ("When are live classes?", False)]

print("Query Reformulation:")
for fu, is_first in turns:
    standalone = reformulate(fu, history)
    raw_doc, raw_s = retrieve(fu); ref_doc, ref_s = retrieve(standalone)
    print(f"  FU: {fu}")
    if standalone != fu: print(f"  -> {standalone[:50]}...")
    print(f"  Raw:[{raw_s}] {raw_doc[:30]}.. Ref:[{ref_s}] {ref_doc[:30]}.. {'IMPROVED' if ref_s > raw_s else ''}")
    history.append({"role":"user","content":fu}); history.append({"role":"assistant","content":ref_doc[:40]})
print(f"\nWithout reformulation: pronouns -> random retrieval")


</details>


---
## Exercise 4 (Medium): Memory Architecture Comparison


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class Mem:
    @staticmethod
    def buffer(m): return {"count":len(m),"tokens":sum(len(x.split()) for x in m),"lost":0}
    @staticmethod
    def window(m, k=6): kept=m[-k:]; return {"count":len(kept),"tokens":sum(len(x.split()) for x in kept),"lost":len(m)-len(kept)}
    @staticmethod
    def summary(m, k=6):
        if len(m)<=k: return {"count":len(m),"tokens":sum(len(x.split()) for x in m),"lost":0}
        old=m[:-k]; recent=m[-k:]; w=" ".join(old).lower().split()
        t=[x for x in set(w) if len(x)>4 and w.count(x)>1][:4]; s=f"Earlier: {', '.join(t)}"
        return {"count":len(recent)+1,"tokens":len(s.split())+sum(len(x.split()) for x in recent),"lost":0}
    @staticmethod
    def hybrid(m, k=6):
        if len(m)<=k: return {"count":len(m),"tokens":sum(len(x.split()) for x in m),"lost":0}
        old=m[:-k]; recent=m[-k:]; w=" ".join(old).lower().split()
        t=[x for x in set(w) if len(x)>4 and w.count(x)>1][:4]; s=f"Summary: {', '.join(t)}"
        return {"count":len(recent)+1,"tokens":len(s.split())+sum(len(x.split()) for x in recent),"lost":0}

turns=[]
for i in range(20):
    t=["courses","pricing","refund","emi","prereqs","classes","credits","cert","placement","discord"][i%10]
    turns.append(f"User: Tell me about {t} at Netsetos in detail"); turns.append(f"Bot: Info about {t} with details")
m=Mem()
print("Memory Comparison (20 turns):")
for name, r in [("Buffer",m.buffer(turns)),("Window(k=6)",m.window(turns)),("Summary",m.summary(turns)),("Hybrid",m.hybrid(turns))]:
    print(f"  {name:<14} msgs={r['count']:<4} tokens={r['tokens']:<6} lost={r['lost']}")
print(f"\nRecommended: Hybrid (recent verbatim + older summarized)")
print(f"Budget: sys 15%, hist 20%, retrieval 40%, response 25%. 80% ceiling.")


</details>


---
## Exercise 5 (Medium): Token Budget Manager


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class Budget:
    def __init__(self, mx, util=0.80):
        self.mx=mx; self.b=int(mx*util)
        self.a={"sys":int(self.b*0.15),"hist":int(self.b*0.20),"ret":int(self.b*0.40),"resp":int(self.b*0.25)}
    def est(self, t): return int(len(t.split())*1.3)
    def fit(self, sys, hist, chunks, q):
        st=self.est(sys); qt=self.est(q); avail=self.b-st-qt
        hb=min(self.a["hist"],avail//2); kh=[]; ht=0
        for m in reversed(hist):
            mt=self.est(m)
            if ht+mt<=hb: kh.insert(0,m); ht+=mt
            else: break
        cb=min(self.a["ret"],avail-ht); kc=[]; ct=0
        for c in chunks:
            t2=self.est(c)
            if ct+t2<=cb: kc.append(c); ct+=t2
            else: break
        used=st+ht+ct+qt; return {"hist":len(kh),"chunks":len(kc),"tokens":used,"util":used/self.mx*100}

for name, mx in [("8K",8192),("128K",131072)]:
    b=Budget(mx)
    r=b.fit("You are support."*3, [f"Turn {i}: Q&A" for i in range(15)], [f"Chunk {i}: pricing refund info" for i in range(10)], "Cost?")
    print(f"{name}: hist={r['hist']}/15 chunks={r['chunks']}/10 tok={r['tokens']} util={r['util']:.1f}%")
print(f"\nOverflow: summarize hist -> reduce chunks -> NEVER truncate system")


</details>


---
## Exercise 7 (Challenge): Multi-Turn Evaluation


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class MTE:
    @staticmethod
    def faith(a,c): aw=set(a.lower().split()); cw=set(c.lower().split()); return len(aw&cw)/max(len(aw),1)
    @staticmethod
    def rel(q,c): qw=set(q.lower().split()); cw=set(c.lower().split()); return len(qw&cw)/max(len(qw),1)
    @staticmethod
    def drift(rels): return len(rels)>=3 and sum(1 for i in range(1,len(rels)) if rels[i]<rels[i-1]-0.1)>len(rels)//2

ev=MTE()
convs=[{"name":"Clean","turns":[
    {"q":"Refund?","c":"Full refund 7 days 50% 8-30","a":"Full refund 7 days 50% 8-30"},
    {"q":"Process time?","c":"Processed 5 business days","a":"Processed 5 business days"},
    {"q":"By email?","c":"Email support@netsetos.com","a":"Yes email support@netsetos.com"}]},
  {"name":"Drift","turns":[
    {"q":"Pricing?","c":"GenAI 14999 rupees lifetime","a":"GenAI costs 14999 rupees"},
    {"q":"Discounts?","c":"Live classes 7 PM IST","a":"Classes at 7 PM"},
    {"q":"Student discount?","c":"Prerequisites Python math","a":"Need Python"}]}]

print("Multi-Turn Evaluation:")
for conv in convs:
    print(f"\n  [{conv['name']}]")
    rels=[]
    for i, t in enumerate(conv["turns"]):
        f=ev.faith(t["a"],t["c"]); r=ev.rel(t["q"],t["c"]); rels.append(r)
        print(f"    Turn {i+1}: faith={f:.2f} rel={r:.2f} [{'PASS' if f>0.3 else 'FAIL'}]")
    if ev.drift(rels): print(f"    ISSUE: Context drift!")
    else: print(f"    All checks passed")
print(f"\n3 failures: drift, redundant retrieval, cross-turn hallucination")
print(f"Golden: 50-200 multi-turn Q&A pairs. DeepEval CI/CD gates.")


</details>


---
## Exercise 2 (Easy): LlamaIndex ChatEngine
Architecture/design. See practice-lab-8_8.html.


In [ ]:
# Exercise 2: LlamaIndex ChatEngine
pass


---
## Exercise 3 (Medium): LangChain LCEL Conversational RAG
Architecture/design. See practice-lab-8_8.html.


In [ ]:
# Exercise 3: LangChain LCEL Conversational RAG
pass


---
## Exercise 6 (Challenge): LangGraph Stateful RAG
Architecture/design. See practice-lab-8_8.html.


In [ ]:
# Exercise 6: LangGraph Stateful RAG
pass


---
## Exercise 8 (Challenge): India Multilingual Chatbot
Architecture/design. See practice-lab-8_8.html.


In [ ]:
# Exercise 8: India Multilingual Chatbot
pass
